# OpenAlex corpus

Exploration and checks. The ingest itself runs from `scripts/ingest.py`; this notebook imports its functions so there is only one copy of each.

## Setup

In [25]:
import sys

import duckdb

sys.path.insert(0, "../scripts")
from ingest import corpus, works_url, paginate, year_params, reconstruct_abstract, flatten_work

In [26]:
assert reconstruct_abstract({"The": [0], "cat": [1], "sat": [2]}) == ("The cat sat", 0)
assert reconstruct_abstract({"The": [0], "cat": [1], "sat": [3]}) == ("The cat sat", 1)
assert reconstruct_abstract(None) == (None, 0)
assert reconstruct_abstract({}) == (None, 0)

## Sanity checks on one page

In [27]:
pages = paginate(works_url, year_params(2010), per_page=25)
first_page = next(pages)
print("Number of results:", first_page["meta"]["count"])

Number of results: 93801


In [28]:
for work in first_page["results"]:
    print(work["title"])

Food Security: The Challenge of Feeding 9 Billion People
Multivariate data analysis : a global perspective
Multiple imputation using chained equations: Issues and guidance for practice
Fast Stable Restricted Maximum Likelihood and Marginal Likelihood Estimation of Semiparametric Generalized Linear Models
A statistical explanation of MaxEnt for ecologists
Picante: R tools for integrating phylogenies and ecology
The value of estuarine and coastal ecosystem services
Synthetic Control Methods for Comparative Case Studies: Estimating the Effect of California’s Tobacco Control Program
Matching Methods for Causal Inference: A Review and a Look Forward
MCMC Methods for Multi-Response Generalized Linear Mixed Models: The MCMCglmm R Package
A distance‐based framework for measuring functional diversity from multiple traits
A general approach to causal mediation analysis.
Optimal caliper widths for propensity‐score matching when estimating differences in means and differences in proportions in obs

In [29]:
print(first_page["results"][0].keys())

dict_keys(['id', 'doi', 'title', 'authorships', 'cited_by_count', 'publication_year', 'publication_date', 'primary_topic', 'referenced_works', 'abstract_inverted_index', 'language'])


In [30]:
[reconstruct_abstract(work["abstract_inverted_index"]) for work in first_page["results"][:5]]
# All abstracts read normally, seems to work okay

[('Continuing population and consumption growth will mean that the global demand for food will increase for at least another 40 years. Growing competition for land, water, and energy, in addition to the overexploitation of fisheries, will affect our ability to produce food, as will the urgent requirement to reduce the impact of the food system on the environment. The effects of climate change are a further threat. But the world can produce more food and can ensure that it is used more efficiently and equitably. A multifaceted and linked global strategy is needed to ensure sustainable and equitable food security, different components of which are explored here.',
  0),
 ('1 Introduction: Models and Model Building Section I Understanding and Preparing for Multivariate Analysis 2 Cleaning and Transforming Data 3 Factor Analysis Section II Analysis Using Dependece Techniques 4 Simple and Multiple Regression Analysis 5 Canonical correlation 6 Conjoint analysis 7 Multiple Discriminant Analys

## Database

Read-only, so nothing here can change the data. Close the connection (last cell) before running the ingest script.

In [31]:
con = duckdb.connect(corpus["db_path"], read_only=True)

In [32]:
print(con.sql("SHOW TABLES"))
print(con.sql("DESCRIBE works"))
print(con.sql("DESCRIBE work_authors"))
print(con.sql("DESCRIBE work_references"))

┌─────────────────┐
│      name       │
│     varchar     │
├─────────────────┤
│ work_authors    │
│ work_references │
│ works           │
└─────────────────┘

┌────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│    column_name     │ column_type │  null   │   key   │ default │  extra  │
│      varchar       │   varchar   │ varchar │ varchar │ varchar │ varchar │
├────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ work_id            │ VARCHAR     │ NO      │ PRI     │ NULL    │ NULL    │
│ doi                │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ title              │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ abstract           │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ abstract_gap_count │ INTEGER     │ YES     │ NULL    │ NULL    │ NULL    │
│ publication_year   │ INTEGER     │ YES     │ NULL    │ NULL    │ NULL    │
│ publication_date   │ DATE        │ YES     │ NULL    │ NULL    │ NU

## Check flatten_work against the tables

In [33]:
work_columns = set(con.sql("DESCRIBE works").df()["column_name"])
author_columns = set(con.sql("DESCRIBE work_authors").df()["column_name"])
reference_columns = set(con.sql("DESCRIBE work_references").df()["column_name"])

n_authors = 0
n_references = 0
n_null_author_ids = 0

for work in first_page["results"]:
    work_row, author_rows, reference_rows = flatten_work(work)
    work_id = work_row["work_id"]

    # The work row has exactly the columns of the works table
    assert set(work_row) == work_columns, work_id

    # One author row per authorship, numbered from 1, all pointing back to this work
    assert len(author_rows) == len(work["authorships"]), work_id
    for row in author_rows:
        assert set(row) == author_columns, work_id
        assert row["work_id"] == work_id, work_id
    if author_rows:
        assert author_rows[0]["author_order"] == 1, work_id

    # One reference row per referenced work, all pointing back to this work
    assert len(reference_rows) == len(work["referenced_works"]), work_id
    for row in reference_rows:
        assert set(row) == reference_columns, work_id
        assert row["citing_work_id"] == work_id, work_id

    n_authors += len(author_rows)
    n_references += len(reference_rows)
    n_null_author_ids += sum(row["author_id"] is None for row in author_rows)

print(f"All checks passed: {len(first_page['results'])} works, {n_authors} author rows, "
      f"{n_references} reference rows, {n_null_author_ids} author rows with no author ID")

All checks passed: 25 works, 99 author rows, 1413 reference rows, 3 author rows with no author ID


## Checks on the ingested data

In [34]:
checks = {
    "Works (compare with the counts the script printed)":
        "SELECT count(*) AS n_works FROM works",
    "Works per year":
        "SELECT publication_year, count(*) AS n FROM works GROUP BY publication_year ORDER BY publication_year",
    "Shares on works (sample: ~0.15 null DOI, ~0.58 no references, ~0.01 gaps)":
        """SELECT
               avg((doi IS NULL)::INTEGER) AS null_doi_share,
               avg((work_id NOT IN (SELECT citing_work_id FROM work_references))::INTEGER) AS no_reference_share,
               avg((abstract_gap_count > 0)::INTEGER) AS gap_share
           FROM works""",
    "Author rows with no author ID (sample: ~0.11)":
        "SELECT avg((author_id IS NULL)::INTEGER) AS null_author_id_share FROM work_authors",
    "Duplicate reference pairs (should be 0)":
        """SELECT count(*) AS duplicate_pairs
           FROM (SELECT citing_work_id, cited_work_id
                 FROM work_references
                 GROUP BY citing_work_id, cited_work_id
                 HAVING count(*) > 1)""",
    "Most common languages":
        "SELECT language, count(*) AS n FROM works GROUP BY language ORDER BY n DESC LIMIT 10",
    "Works per subfield":
        "SELECT subfield_name, count(*) AS n FROM works GROUP BY subfield_name ORDER BY n DESC",
    "Most common topics":
        "SELECT topic_name, count(*) AS n FROM works GROUP BY topic_name ORDER BY n DESC LIMIT 15",
    "Works in 'Species Distribution and Climate Change' (T10895)":
        "SELECT count(*) AS n FROM works WHERE topic_id = 'T10895'",
}

for label, query in checks.items():
    print(label)
    con.sql(query).show()

Works (compare with the counts the script printed)
┌─────────┐
│ n_works │
│  int64  │
├─────────┤
│   93801 │
└─────────┘

Works per year
┌──────────────────┬───────┐
│ publication_year │   n   │
│      int32       │ int64 │
├──────────────────┼───────┤
│             2010 │ 93801 │
└──────────────────┴───────┘

Shares on works (sample: ~0.15 null DOI, ~0.58 no references, ~0.01 gaps)
┌────────────────────┬────────────────────┬──────────────────────┐
│   null_doi_share   │ no_reference_share │      gap_share       │
│       double       │       double       │        double        │
├────────────────────┼────────────────────┼──────────────────────┤
│ 0.4361680579098304 │ 0.5303354974893658 │ 0.020564812741868423 │
└────────────────────┴────────────────────┴──────────────────────┘

Author rows with no author ID (sample: ~0.11)
┌──────────────────────┐
│ null_author_id_share │
│        double        │
├──────────────────────┤
│  0.04780053531944606 │
└──────────────────────┘

Duplicate re

In [35]:
print("--- Random sample ---")
for title, abstract in con.sql("SELECT title, abstract FROM works USING SAMPLE 5").fetchall():
    print(title, "\n", abstract, "\n")

print("--- Shortest abstracts ---")
for title, abstract, n_chars in con.sql(
        "SELECT title, abstract, length(abstract) AS n_chars FROM works ORDER BY n_chars LIMIT 5").fetchall():
    print(n_chars, "|", title, "|", abstract)

--- Random sample ---
Inter-population segregation in the wintering areas of macaroni penguins 
 How top avian predators are distributed at sea during the inter-breeding period remains poorly known.Here we focused on the pelagic seabird that is the single greatest avian consumer of marine resources: the macaroni penguin.Our aims were (1) to investigate where these penguins are distributed whilst at sea during winter, and (2) to determine how their distribution and trophic ecology varies in 1 colony between successive years and between 2 colonies from neighbouring localities at the same time.We surveyed a total of 30 penguins from Crozet and Kerguelen Islands (southern Indian Ocean).Penguins from Kerguelen were tracked in 2006 and 2007, and those from Crozet in 2007.Habitat use was investigated using miniaturised light-based geolocators, and trophic ecology by the use of stable isotope analyses.Results showed large-scale patterns of distribution over deep oceanic waters of the Polar Fro

## Close

In [36]:
con.close()  # release the database file so the ingest script can write to it